# Bodhan Tool Server and Autonomous Task Agent Walkthrough

This notebook demonstrates using Bodhan Indic models as Model Context Protocol (MCP) tools and orchestrating them through an autonomous CLI Task Agent.

Models wrapped:
- `indic-transcribe`: Speech-to-text with automatic 30s chunking.
- `indic-translate`: 25 Indic languages + English, with automatic English pivot for Indic-to-Indic.
- `indic-speak`: Text-to-speech with sentence tokenization and WAV stitching.
- `indic-ocr`: Document OCR supporting multi-page PDFs and images.

In [ ]:
%pip install -q -r requirements.txt

## 1. Inspecting Media without Model Quota

Use `run_inspect_media` to check file types, audio durations, PDF pages, or detect Indian language scripts.

In [ ]:
from bodhan_mcp.tools.inspect_tool import run_inspect_media

inspect_res = run_inspect_media("sample_data/hindi_voice_note.wav")
print("Audio Inspection:", inspect_res)

## 2. Speech-to-Text with `indic-transcribe`

In [ ]:
from bodhan_mcp.tools.transcribe import run_transcribe

asr_res = run_transcribe("sample_data/hindi_voice_note.wav", language="hi")
print("Transcript:", asr_res.get("text"))
print(f"Cost: ₹{asr_res.get('cost_inr')} INR")

## 3. Translation with `indic-translate`

Translate between any of the 25 Indic languages and English.

In [ ]:
from bodhan_mcp.tools.translate import run_translate

trans_res = run_translate("Water boils at 100 degrees Celsius.", target_lang="ta")
print("Tamil Translation:", trans_res.get("translation"))

## 4. Speech Synthesis with `indic-speak`

Generates natural PCM16 WAV audio on disk and returns the file path reference.

In [ ]:
from bodhan_mcp.tools.speak import run_speak

tts_res = run_speak("வணக்கம், இது போதன் ஏஐயின் குரல்.", language="ta", style="educational lecture")
print("Generated Audio Path:", tts_res.get("audio_file"))

## 5. Running the Autonomous Task Agent

The Task Agent plans and executes multi-step tasks end-to-end, enforcing budgets and output verification.

In [ ]:
from bodhan_agent.agent import TaskAgent

agent = TaskAgent(
    task="Make a two-minute Tamil audio summary of this chapter",
    files=["sample_data/sample_chapter.pdf"],
    out_dir="outputs/walkthrough_run",
    max_steps=10,
    max_cost_inr=5.0,
)
summary = agent.run()
print("Execution status:", summary["status"])
print("Artifacts:", summary["artifacts"])
print("Total cost:", f"₹{summary['total_cost_inr']} INR")